# 13.11 拿到分數後，為什麼還要扣掉預期？


一次選卡拿1分，另一次拿0分。若這種情境平常預期0.4分，兩次相對預期分別是0.6與−0.4。我們希望策略模型更常作出比預期好的選擇、較少作出比預期差的選擇。若直接拿獎勵決定鼓勵多少，0分那次的鼓勵是0，它自己不會帶來任何調整；扣掉預期後是−0.4，才會把這次選擇調低。

選卡的**策略模型（policy）**和[13.4](https://birdhackor.github.io/tiny-perceptron-vlm/13.4.html)是同一種東西：讀入情境，輸出每個動作的機率。這裡的情境是13.10的四個數，動作是選四張回答卡中的哪一張；[13.15](https://birdhackor.github.io/tiny-perceptron-vlm/13.15.html)會看到它就是一個小網路。估計作答前平均會拿多少分，叫**基準值（baseline）**。實際獎勵減更新前的基準值，叫**優勢（advantage）**。正負說的是高於或低於預期，不直接等於答案對錯。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.posttraining import bandit_advantage

rewards = torch.tensor([1.0, 0.0])
old_values = torch.tensor([0.4, 0.4], requires_grad=True)
advantage = bandit_advantage(rewards, old_values)
print("優勢", [round(value, 2) for value in advantage.tolist()])
print("更新策略時還追蹤基準值的梯度嗎", advantage.requires_grad)

輸出`[0.6,−0.4]`。`old_values`就是更新前的基準值；這裡故意打開梯度追蹤，模仿它由另一個會學的模型算出；最後印出`False`，表示優勢已被當成固定數值。`bandit_advantage`的寫法如下：


In [ ]:
# 節錄自 tiny_perceptron/posttraining.py：bandit_advantage
def bandit_advantage(rewards, old_values):
    """單一步驟任務 A = R - V_old；本輪更新時兩份採樣資料保持固定。"""
    if rewards.shape != old_values.shape or not rewards.numel():
        raise ValueError("回饋與舊 value 必須同形狀且不可為空")
    return (rewards - old_values).detach()

說明文字裡的`A = R - V_old`，就是「優勢＝獎勵−更新前的基準值」。`if`那行檢查兩份資料同形狀、而且不是空的，否則`raise ValueError`停下。最後一行相減後接`.detach()`：把結果當成固定數字，更新策略模型時不會沿這條計算去改基準值；算基準值的模型另用自己的預測代價學習（[13.14](https://birdhackor.github.io/tiny-perceptron-vlm/13.14.html)）。

本章一次選整張預寫的回答卡，拿到一次獎勵就結束，是**依情境做一次選擇的任務（contextual bandit）**。整張卡只得一次分數，這個分數直接屬於這一次選擇，所以「獎勵減基準值」就是這次選擇的優勢。逐token寫答案則是一串選擇：早期的字會影響後文與最後的獎勵，要另外分配每一步的功勞，不能把此例當成完整的語言模型RLHF。

接下來的PPO是一種更新方法：它使用固定的優勢，又看新舊策略給同一個選擇的機率比（[13.12](https://birdhackor.github.io/tiny-perceptron-vlm/13.12.html)定義），處理同一批舊作答被重用時的變化。獎勵的尺度也須記下，真正的獎勵模型不必每次只給0與1。

練習把`old_values`的兩個0.4改成0.9，先預測優勢變`[0.1,−0.9]`，再執行核對。獎勵沒有改，超出預期的程度改了。
